# evtools — Tutorial

**Evidence Theory Tools** — a Python library for working with belief functions in the Dempster-Shafer / Transferable Belief Model framework.

This tutorial uses a running example: a sensor classifying aerial targets as **airplane** (a), **helicopter** (h), or **rocket** (r).

**Sections**
1. Building a BBA
2. Accessing values
3. Conversions (bel, pl, b, q)
4. Conjunctive and disjunctive weights (v, w)
5. Simple MFs
6. Combination rules
7. Decombination
8. Correction mechanisms
9. Display formats
10. display_all — all representations in one table
11. Conditioning and deconditioning
12. Pignistic and plausibility probability transformations
13. Low-level conversions API

**References**
- P. Smets. *The application of the matrix calculus to belief functions*, IJAR, 31(1–2):1–30, 2002.
- T. Denœux. *Conjunctive and disjunctive combination of belief functions induced by nondistinct bodies of evidence*, AI, 172:234–264, 2008.
- D. Mercier, B. Quost, T. Denœux. *Refined modeling of sensor reliability in the belief function framework using contextual discounting*, Information Fusion, 9(2):246–258, 2008.
- F. Pichon, D. Mercier, É. Lefèvre, F. Delmotte. *Proposition and learning of some belief function contextual correction mechanisms*, IJAR, 72:4–42, 2016.

## Setup

In [ ]:
import numpy as np
from evtools.dsvector import DSVector, Kind
from evtools.combinations import crc, dempster, drc, cautious, bold, decombine_crc, decombine_drc
from evtools.corrections import (
    discount, contextual_discount, theta_contextual_discount,
    contextual_reinforce, contextual_dediscount, contextual_dereinforce,
    contextual_negate,
)
from evtools.display import repr_plain, repr_html, repr_latex

frame = ["a", "h", "r"]  # airplane, helicopter, rocket
print("evtools loaded successfully")

---
## 1. Building a Basic Belief Assignment (BBA)

A BBA is a function $m: 2^\Omega \to [0,1]$ with $\sum_{A \subseteq \Omega} m(A) = 1$.

**Three constructors are available:**

In [ ]:
# from_focal: human-friendly string keys.
# Missing mass is automatically assigned to Ω = {a, h, r}.
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})
m  # Jupyter renders the HTML table automatically via _repr_html_

In [ ]:
# from_dense: numpy array in binary index order (Smets 2002).
# Index i → subset whose members are the atoms at the bit positions set in i.
# For frame=[a,h,r]: 0=∅, 1={a}, 2={h}, 3={a,h}, 4={r}, 5={a,r}, 6={h,r}, 7={a,h,r}
array = np.array([0.0, 0.5, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0])
m2 = DSVector.from_dense(frame, array)
m2

In [ ]:
# from_sparse: dict of frozensets.
m3 = DSVector.from_sparse(frame, {
    frozenset({"a"}): 0.5,
    frozenset({"r"}): 0.5,
})
m3

In [ ]:
# Subnormal BBA: m(∅) > 0 (allowed in the TBM, represents internal conflict)
m_sub = DSVector.from_focal(frame, {"": 0.1, "a": 0.3, "r": 0.4, "a,h,r": 0.2}, complete=False)
print(f"is_valid: {m_sub.is_valid}")
m_sub

---
## 2. Accessing Values

In [ ]:
print("sparse dict :", m.sparse)
print("dense array :", m.dense)
print("m({a})      :", m[frozenset({"a"})])
print("m({h})      :", m[frozenset({"h"})], " ← not a focal element")
print("n_focal     :", m.n_focal)
print("is_valid    :", m.is_valid)

In [ ]:
print("Iterating over focal elements:")
for subset, value in m:
    label = "{"+", ".join(sorted(subset))+"}" if subset else "∅"
    print(f"  m({label}) = {value:.4f}")

---
## 3. Conversions

All standard representations are available via `.to(Kind)` or shortcuts.
The column header in the display adapts to the kind (`m`, `bel`, `pl`, `b`, `q`, `v`, `w`).

In [ ]:
m.to_bel()  # Belief function

In [ ]:
m.to_pl()  # Plausibility function

In [ ]:
m.to_b()  # Commonality function

In [ ]:
m.to_q()  # Implicability function

In [ ]:
# Round-trip consistency check
for kind in [Kind.BEL, Kind.PL, Kind.B, Kind.Q]:
    back = m.to(kind).to(Kind.M)
    ok = np.allclose(back.dense, m.dense, atol=1e-10)
    print(f"  m → {kind.value:>3} → m : {'✓ OK' if ok else '✗ MISMATCH'}")

---
## 4. Conjunctive and Disjunctive Weights (v, w)

The conjunctive weight function $w$ and disjunctive weight function $v$ require
a **subnormal BBA** ($m(\emptyset) > 0$), so that $b(A) > 0$ for all $A \subseteq \Omega$.

They are defined via a Möbius transform on $\ln q$ (for $w$) and $\ln b$ (for $v$),
and are essential for the **Cautious** and **Bold** combination rules (Denoeux 2008).

In [ ]:
# Subnormal BBA required: m(∅) > 0
m_sub2 = DSVector.from_focal(frame, {"": 0.1, "a": 0.3, "r": 0.4, "a,h,r": 0.2}, complete=False)
print(f"is_valid: {m_sub2.is_valid}")
m_sub2

In [ ]:
# Disjunctive weight function v
m_sub2.to_v()

In [ ]:
# Conjunctive weight function w
m_sub2.to_w()

In [ ]:
# Round-trip consistency for v and w
for kind in [Kind.V, Kind.W]:
    back = m_sub2.to(kind).to(Kind.M)
    ok = np.allclose(back.dense, m_sub2.dense, atol=1e-10)
    print(f"  m_sub → {kind.value} → m : {'✓ OK' if ok else '✗ MISMATCH'}")

---
## 5. Simple MFs

Simple MFs are the elementary building blocks of correction mechanisms:
- $A^\beta$ (`DSVector.simple`): focal sets $\Omega$ (mass $\beta$) and $A$ (mass $1-\beta$) — used in CR, CdR, CN
- $A_\beta$ (`DSVector.negative_simple`): focal sets $\emptyset$ (mass $\beta$) and $A$ (mass $1-\beta$) — used in CD, CdD

In [ ]:
# Simple MF A^β (positive)
DSVector.simple(frame, frozenset({"a"}), beta=0.6)

In [ ]:
# Negative simple MF A_β
DSVector.negative_simple(frame, frozenset({"a"}), beta=0.4)

---
## 6. Combination Rules

| | All sources reliable | At least one reliable |
|---|---|---|
| **Distinct sources** | `crc` (`&`) / `dempster` (`@`) | `drc` (`\|`) |
| **Nondistinct sources** | `cautious` | `bold` |

In [ ]:
s1 = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})
s2 = DSVector.from_focal(frame, {"h": 0.3, "r": 0.4, "a,h,r": 0.3})

# CRC — distinct, both reliable (operator &)
m12_crc = s1 & s2
print(f"Conflict m(∅) = {m12_crc[frozenset()]:.4f}")
m12_crc

In [ ]:
# Dempster's rule — normalized CRC (operator @)
s1 @ s2

In [ ]:
# DRC — at least one reliable (operator |)
s1 | s2

In [ ]:
# Cautious — nondistinct, both reliable; commutative, associative, idempotent
s1_nd = DSVector.from_focal(frame, {"a": 0.3, "h": 0.2, "a,h,r": 0.5})
s2_nd = DSVector.from_focal(frame, {"a": 0.4, "r": 0.1, "a,h,r": 0.5})
m_caut = cautious(s1_nd, s2_nd)
print(f"Idempotent: cautious(m,m)==m → {np.allclose(cautious(s1_nd,s1_nd).dense, s1_nd.dense, atol=1e-10)}")
m_caut

In [ ]:
# Bold — nondistinct, at least one reliable (requires subnormal BBAs)
b1 = DSVector.from_focal(frame, {"": 0.1, "a": 0.4, "a,h,r": 0.5}, complete=False)
b2 = DSVector.from_focal(frame, {"": 0.2, "r": 0.3, "a,h,r": 0.5}, complete=False)
bold(b1, b2)

---
## 7. Decombination

Inverse operations to remove a previously combined BBA.
The result may not be a valid BBA — always check `.is_valid`.

In [ ]:
# decombine_crc: m1 6∩ m2
m1 = DSVector.from_focal(frame, {"a": 0.4, "a,h,r": 0.6})
m2 = DSVector.from_focal(frame, {"h": 0.3, "a,h,r": 0.7})
m12 = crc(m1, m2)
m1_recovered = decombine_crc(m12, m2)

print(f"is_valid  : {m1_recovered.is_valid}")
print(f"Recovers m1: {np.allclose(m1_recovered.dense, m1.dense, atol=1e-6)}")
m1_recovered

---
## 8. Correction Mechanisms

Corrections adjust a BBA based on knowledge about the source quality (reliability, truthfulness).

| Mechanism | Notation | Source assumption |
|-----------|---------|-------------------|
| `discount(m, β)` | $m_S \cup \Omega_{\beta}$ | single reliability degree |
| `contextual_discount(m, β)` (CD) | $m_S \cup \bigcup_{A} A_{\beta_A}$ | reliability per singleton |
| `theta_contextual_discount(m, β)` | general Θ partition | reliability per coarsening |
| `contextual_reinforce(m, β)` (CR) | $m_S \cap \bigcap_{A} A^{\beta_A}$ | contextual positive liar |
| `contextual_dediscount(m, β)` (CdD) | inverse of CD | undo a prior CD |
| `contextual_dereinforce(m, β)` (CdR) | inverse of CR | undo a prior CR |
| `contextual_negate(m, β)` (CN) | equivalence rule | contextual non-truthful |

In [ ]:
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})

# Classical discounting: β=0.6, source 60% reliable
discount(m, beta=0.6)

In [ ]:
# Contextual discounting: sensor unreliable only when target is airplane
# β_a=0.6, β_h=1.0, β_r=1.0  (Example 1, Case 1 of Mercier et al. 2008)
betas_cd = {frozenset({"a"}): 0.6, frozenset({"h"}): 1.0, frozenset({"r"}): 1.0}
mcd = contextual_discount(m, betas_cd)
# Interpretation: mass on {r} partially transferred to {a,r}
# because if the true target is airplane, sensor may declare it a rocket
mcd

In [ ]:
# Θ-contextual discounting: coarser partition Θ = {{a}, {h,r}}
theta_contextual_discount(m, {frozenset({"a"}): 0.4, frozenset({"h","r"}): 0.9})

In [ ]:
# Contextual Reinforcement (CR) — dual of CD, uses CRC
contextual_reinforce(m, betas_cd)

In [ ]:
# CdD: inverse of CD — check .is_valid after decombination
mdd = contextual_dediscount(mcd, betas_cd)
print(f"is_valid      : {mdd.is_valid}")
print(f"CdD reverts CD: {np.allclose(mdd.dense, m.dense, atol=1e-6)}")
mdd

In [ ]:
# Contextual Negating (CN): source non-truthful with probability 1−β
contextual_negate(m, {frozenset({"a"}): 0.7})

In [ ]:
# Pure negation: β=0, A=∅  →  m(B) becomes m(B̄) for all B
contextual_negate(m, {frozenset(): 0.0})

---
## 9. Display Formats

Four output formats are available. In Jupyter, `DSVector` renders as an HTML table automatically (via `_repr_html_`). Use `.display(fmt)` to request a specific format.

In [ ]:
m = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})

# HTML (default in Jupyter)
m

In [ ]:
# Plain text — no colors, for logging and files
print(repr_plain(m))

In [ ]:
# Column header adapts to the kind
print("--- Belief function ---")
print(repr_plain(m.to_bel()))
print("\n--- Plausibility function ---")
print(repr_plain(m.to_pl()))
print("\n--- Implicability function ---")
print(repr_plain(m.to_q()))

In [ ]:
# LaTeX — ready to paste in a paper
print(repr_latex(m))

In [ ]:
# Explicit format selection
print(m.display("plain"))

---
## 13. Low-level Conversions API

All conversions are available as standalone functions on numpy arrays,
using the Fast Möbius Transform (Smets 2002, Section 3).

In [ ]:
from evtools.conversions import mtob, mtopl, mtoq, mtobel

m_array = np.array([0.0, 0.5, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0])
print("m     :", m_array)
print("bel   :", mtobel(m_array))
print("pl    :", mtopl(m_array))
print("b     :", mtob(m_array))
print("q     :", mtoq(m_array))

---
## 10. display_all — All Representations in One Table

`display_all(m)` renders `m`, `bel`, `pl`, `b`, `q` in a single table.
Additional columns are added automatically:

- **v** (disjunctive weights): added when the BBA is **subnormal** ($m(\emptyset) > 0$),
  so that $b(A) > 0$ for all $A \subseteq \Omega$
- **w** (conjunctive weights): added when the BBA is **non-dogmatic** ($m(\Omega) > 0$),
  so that $q(A) > 0$ for all $A \subsetneq \Omega$

In [ ]:
from evtools.display import display_all

# Normal dogmatic BBA: m(∅)=0, m(Ω)=0 → only m, bel, pl, b, q
m_dog = DSVector.from_focal(frame, {"a": 0.5, "r": 0.5})
print("Normal dogmatic BBA (no v, no w):")
print(repr_plain(m_dog))

In [ ]:
# Non-dogmatic BBA: m(Ω)>0 → w column added
m_nd = DSVector.from_focal(frame, {"a": 0.3, "r": 0.3, "a,h,r": 0.4})
print(f"Non-dogmatic (m(Ω)={m_nd[frozenset({'a','h','r'})]:.1f}) → w column added:")
print(display_all(m_nd, "plain"))

In [ ]:
# Subnormal dogmatic: m(∅)>0, m(Ω)=0 → v column added
m_sub3 = DSVector.from_focal(frame, {"": 0.1, "a": 0.5, "r": 0.4}, complete=False)
print(f"Subnormal dogmatic (m(∅)={m_sub3[frozenset()]:.1f}) → v column added:")
print(display_all(m_sub3, "plain"))

In [ ]:
# Subnormal non-dogmatic: m(∅)>0 and m(Ω)>0 → both v and w
m_full = DSVector.from_focal(frame, {"": 0.1, "a": 0.3, "r": 0.4, "a,h,r": 0.2}, complete=False)
print("Subnormal non-dogmatic → both v and w:")
print(display_all(m_full, "plain"))

In [ ]:
# In Jupyter: display_all renders as HTML
from IPython.display import HTML
HTML(display_all(m_nd, "html"))

In [ ]:
# LaTeX output — ready to paste in a paper
print(display_all(m_nd, "latex"))

---
## 11. Conditioning and Deconditioning

**Conditioning** on $A$ gives the least committed **specialization** of $m$
such that $pl(\bar{A}) = 0$ (Smets 2002, Section 9):

$$m[A](B) = \sum_{C\,:\,C\cap A = B} m(C), \quad \forall B \subseteq A$$

**Deconditioning** is the inverse: the least committed **generalization** $m^*$
such that $pl^*(\bar{A}) = 1$:

$$m^*(B \cup \bar{A}) = m(B), \quad \forall B \subseteq A$$

Both are available in `sparse` (default) and `dense` modes via the
matrices $C_A$ and $D_A$ (`conversions.conditioning_matrix` / `deconditioning_matrix`).

In [ ]:
from evtools.combinations import condition, decondition
from evtools.conversions import conditioning_matrix, deconditioning_matrix

m_base = DSVector.from_focal(frame, {"a": 0.3, "h": 0.2, "a,h": 0.1, "a,h,r": 0.4})
A = frozenset({"a", "h"})
print("Original BBA:")
m_base

In [ ]:
# Conditioning on A = {a, h}
m_cond = condition(m_base, A)
print("m conditioned on A = {a, h}:")
m_cond

In [ ]:
# Deconditioning: B → B ∪ Ā = B ∪ {r}
m_decond = decondition(m_cond, A)
print("Deconditioned BBA (B → B ∪ {r}):")
m_decond

In [ ]:
# Round-trip: condition(decondition(m[A], A), A) == m[A]
ok = np.allclose(condition(m_decond, A).dense, m_cond.dense, atol=1e-10)
print(f"condition(decondition(m[A], A), A) == m[A] : {ok}")

# Sparse vs dense
ok2 = np.allclose(
    condition(m_base, A, method='sparse').dense,
    condition(m_base, A, method='dense').dense, atol=1e-10)
print(f"sparse == dense : {ok2}")

In [ ]:
# Conditioning matrix C_A — specialization matrix (column sums = 1)
CA = conditioning_matrix(frame, A)
print(f"C_A shape: {CA.shape}")
print(f"Column sums = 1: {np.allclose(CA.sum(axis=0), 1.0)}")
print(f"C_A @ m == condition(m, A): {np.allclose(CA @ m_base.dense, m_cond.dense, atol=1e-10)}")

In [ ]:
# Deconditioning matrix D_A — generalization matrix (column sums = 1)
DA = deconditioning_matrix(frame, A)
print(f"D_A shape: {DA.shape}")
print(f"Column sums = 1: {np.allclose(DA.sum(axis=0), 1.0)}")
print(f"D_A @ m_cond == decondition(m_cond, A): {np.allclose(DA @ m_cond.dense, m_decond.dense, atol=1e-10)}")

---
## 13. Pignistic and Plausibility Probability Transformations

Both transformations map a BBA to a **probability vector of length $n$**
(one value per atom), used for decision-making in the TBM.

$$\mathrm{BetP}(\{x\}) = \sum_{A \ni x} \frac{m(A)}{|A| \cdot (1 - m(\emptyset))}$$

$$\mathrm{PlP}(\{x\}) = \frac{pl(\{x\})}{\sum_{y \in \Omega} pl(\{y\})}$$

The result is a `numpy.ndarray` of length $n$, **not** a `DSVector`.

In [ ]:
from evtools.conversions import betp, plp

m_demo = DSVector.from_focal(frame, {"a": 0.3, "a,h": 0.4, "a,h,r": 0.3})
m_demo

In [ ]:
# BetP — pignistic transformation
bp = m_demo.to_betp()
print("BetP:")
for atom, val in zip(frame, bp):
    print(f"  BetP({{{atom}}}) = {val:.4f}")
print(f"Sum = {bp.sum():.4f}")
print(f"type: {type(bp)}, length: {len(bp)}  ← n atoms, not 2^n")

In [ ]:
# PlP — plausibility probability
pp = m_demo.to_plp()
print("PlP:")
for atom, val in zip(frame, pp):
    print(f"  PlP({{{atom}}}) = {val:.4f}")
print(f"Sum = {pp.sum():.4f}")

In [ ]:
# Special cases
m_vac = DSVector.from_focal(frame, {})
m_cat = DSVector.from_focal(frame, {"a": 1.0})
print(f"Vacuous BBA → BetP = {m_vac.to_betp().round(4)} (uniform)")
print(f"Categorical m({{a}})=1 → BetP = {m_cat.to_betp().round(4)}")

# BetP undefined for fully contradictory BBA
import numpy as np
m_conflict = DSVector.from_sparse(frame, {frozenset(): 1.0})
try:
    m_conflict.to_betp()
except ValueError as e:
    print(f"Conflict BBA raises: {e}")

---
## 14. Decision criteria

`evtools.decision` provides two families of decision criteria:

- **Complete preference relations** — return the optimal act as `(index, atom)`:
  - `maximin(m, U)` — pessimistic: max lower expected utility
  - `maximax(m, U)` — optimistic: max upper expected utility
  - `pignistic_decision(m, U)` — MEU with BetP (Smets pignistic)
  - `plp_decision(m, U)` — MEU with PlP (Cobb & Shenoy 2006)
  - `probability_decision(m, U, transform=...)` — generic MEU under any m → probability transform (default = `plp`)
  - `hurwicz(m, U, alpha)` — convex combination of maximin and maximax

- **Partial preference relations** — return a `frozenset` of non-dominated atoms:
  - `strong_dominance(m)` — $\omega \succ \omega' \iff Bel(\{\omega\}) \geq Pl(\{\omega'\})$
  - `weak_dominance(m)` — $\omega \succ \omega' \iff Bel(\{\omega\}) \geq Bel(\{\omega'\})$ and $Pl(\{\omega\}) \geq Pl(\{\omega'\})$

The utility matrix $U$ has shape $(n, n)$ with $U[i, j] = u(a_i, \omega_j)$. Default = identity (0-1 utilities).


In [ ]:
from evtools.decision import (
    maximin, maximax, pignistic_decision, plp_decision, probability_decision,
    hurwicz, strong_dominance, weak_dominance,
)
from evtools.conversions import betp, plp

m_dec = DSVector.from_focal(frame, {"a": 0.3, "a,h": 0.4, "a,h,r": 0.3})
m_dec


In [ ]:
# Complete preference relations with default (identity) utility
print(f"maximin              = {maximin(m_dec)}")
print(f"maximax              = {maximax(m_dec)}")
print(f"pignistic_decision   = {pignistic_decision(m_dec)}")
print(f"plp_decision         = {plp_decision(m_dec)}")
print(f"hurwicz(α=0.5)       = {hurwicz(m_dec)}")
print(f"hurwicz(α=1.0)       = {hurwicz(m_dec, alpha=1.0)}  # ≡ maximin")
print(f"hurwicz(α=0.0)       = {hurwicz(m_dec, alpha=0.0)}  # ≡ maximax")


In [ ]:
# Generic MEU: pass the m → probability transform explicitly
print(f"transform=plp  (default) → {probability_decision(m_dec, transform=plp)}")
print(f"transform=betp           → {probability_decision(m_dec, transform=betp)}")

# Bring your own transform (e.g. uniform — ignores m, picks index 0)
import numpy as np
uniform = lambda dense: np.full(len(frame), 1.0 / len(frame))
print(f"transform=uniform        → {probability_decision(m_dec, transform=uniform)}")


In [ ]:
# Custom utility matrix — favor h heavily
U = np.array([[1.0, 0.0, 0.0],
              [0.0, 2.0, 0.0],
              [0.0, 0.0, 3.0]])
print(f"pignistic_decision(m, U) = {pignistic_decision(m_dec, U)}")
print(f"maximin(m, U)            = {maximin(m_dec, U)}")
print(f"maximax(m, U)            = {maximax(m_dec, U)}")


In [ ]:
# Partial preference relations: non-dominated atoms
print(f"strong_dominance(m) = {set(strong_dominance(m_dec))}")
print(f"weak_dominance(m)   = {set(weak_dominance(m_dec))}")


In [ ]:
# Edge cases
m_cat_dec = DSVector.from_focal(frame, {"a": 1.0})
m_vac_dec = DSVector.from_focal(frame, {})
print(f"Categorical {{a}} → pignistic = {pignistic_decision(m_cat_dec)}, "
      f"strong_dom = {set(strong_dominance(m_cat_dec))}")
print(f"Vacuous     → pignistic = {pignistic_decision(m_vac_dec)} (uniform tie-break), "
      f"strong_dom = {set(strong_dominance(m_vac_dec))}")


---
## 15. Performance metrics — `evtools.metrics`

`evtools.metrics` contains evaluation metrics for evidential predictions:

- **Set-valued predictions** (Zaffalon et al. 2012):
  - `discounted_accuracy(d, ω)` — $x = I(\omega \in d)/|d|$
  - `u65(d, ω) = 1.6x - 0.6x^2`
  - `u80(d, ω) = 2.2x - 1.2x^2`
  - `utility_score(d, ω, a=…, b=…)` — generic $a x - b x^2$

- **BBA-valued predictions** (Mercier et al. 2008, Mutmainah 2021):
  - `pl_loss(predictions, labels)` — $\sum_i \sum_k (pl_i(\omega_k) - \delta_{i,k})^2$. Same formula for hard ($E_{pl}$) and soft ($\tilde{E}_{pl}$) labels.
  - `mean_pl_loss(predictions, labels)` — `pl_loss / n`

- **Batch aggregators**: `mean_discounted_accuracy`, `mean_u65`, `mean_u80`, `mean_utility_score`.

- **Hard-classification metrics** (ROC/AUC/accuracy on precise predictions): use `sklearn.metrics` on a probability vector extracted from each BBA via `m.to_betp()` or `m.to_plp()`.

The building block `m.contour()` returns the length-$K$ vector of singleton plausibilities — used internally by `pl_loss`, `strong_dominance`, and `weak_dominance`.


In [ ]:
from evtools.metrics import (
    discounted_accuracy, u65, u80, utility_score,
    mean_discounted_accuracy, mean_u65, mean_u80,
)

true_label = "a"
for d in [frozenset({"a"}), frozenset({"a", "h"}), frozenset({"a", "h", "r"}), frozenset({"r"})]:
    label = str(set(d)) if d else "∅"
    x = discounted_accuracy(d, true_label)
    print(f"d = {label:<22}  x={x:.4f}  u65={u65(d, true_label):.4f}  u80={u80(d, true_label):.4f}")


In [ ]:
# Mean over a paired (predictions, labels) iterable
preds  = [frozenset({"a"}),  frozenset({"a", "h"}), frozenset({"r"})]
labels = ["a",               "a",                    "a"]
print(f"mean_discounted_accuracy = {mean_discounted_accuracy(preds, labels):.4f}")
print(f"mean_u65                 = {mean_u65(preds, labels):.4f}")
print(f"mean_u80                 = {mean_u80(preds, labels):.4f}")


In [ ]:
from evtools.metrics import pl_loss, mean_pl_loss

m_pred1 = DSVector.from_focal(frame, {"a": 0.6, "h": 0.2, "r": 0.2})
m_pred2 = DSVector.from_focal(frame, {"a": 0.5, "h": 0.5})
m_soft  = DSVector.from_focal(frame, {"a": 0.5, "h": 0.5})  # soft label uncertain a vs h

print(f"pl_loss (hard labels)              = {pl_loss([m_pred1, m_pred2], ['a', 'h']):.4f}")
print(f"pl_loss (soft labels)              = {pl_loss([m_pred1, m_pred2], [m_soft, m_soft]):.4f}")
print(f"pl_loss (mixed: 'a' + m_soft)      = {pl_loss([m_pred1, m_pred2], ['a', m_soft]):.4f}")
print(f"mean_pl_loss (hard, n=2)           = {mean_pl_loss([m_pred1, m_pred2], ['a', 'h']):.4f}")

# Building block: contour function (singleton plausibilities)
print(f"m_pred1.contour() = {m_pred1.contour()}")


In [ ]:
# Hard-classification metrics via scikit-learn:
# extract a probability vector (BetP or PlP) from each BBA, then feed to sklearn.
try:
    from sklearn.metrics import accuracy_score, roc_auc_score
    import numpy as np

    # Two BBAs as classifier outputs (true class is 'a' for both)
    m_a = DSVector.from_focal(frame, {"a": 0.7, "a,h": 0.3})
    m_b = DSVector.from_focal(frame, {"a": 0.4, "h": 0.5, "a,h,r": 0.1})
    bba_outputs = [m_a, m_b]
    y_true = ["a", "a"]

    # Hard prediction = argmax BetP
    y_pred = [m.frame[int(np.argmax(m.to_betp()))] for m in bba_outputs]
    print(f"y_pred = {y_pred}, y_true = {y_true}")
    print(f"accuracy_score = {accuracy_score(y_true, y_pred):.4f}")

    # Binary AUC on score = BetP({a})
    score_a = [m.to_betp()[0] for m in bba_outputs]
    y_bin   = [1 if y == "a" else 0 for y in y_true]
    if len(set(y_bin)) == 2:
        print(f"roc_auc_score  = {roc_auc_score(y_bin, score_a):.4f}")
    else:
        print("(roc_auc_score skipped: only one class in y_true; illustrative dataset)")
except ImportError:
    print("(install scikit-learn: `pip install scikit-learn`)")


---
## Summary

| Module | Key objects / functions |
|--------|------------------------|
| `evtools.dsvector` | `DSVector`, `Kind`, `.simple()`, `.negative_simple()`, `.is_valid`, `.display(fmt)` |
| `evtools.combinations` | `crc` (`&`), `dempster` (`@`), `drc` (`\|`), `cautious`, `bold`, `decombine_crc`, `decombine_drc`, `condition`, `decondition` |
| `evtools.corrections` | `discount`, `contextual_discount`, `theta_contextual_discount`, `contextual_reinforce`, `contextual_dediscount`, `contextual_dereinforce`, `contextual_negate` |
| `evtools.decision` | `maximin`, `maximax`, `pignistic_decision`, `plp_decision`, `probability_decision`, `hurwicz`, `strong_dominance`, `weak_dominance` |
| `evtools.metrics` | `discounted_accuracy`, `u65`, `u80`, `utility_score` + `mean_*` aggregators, `pl_loss`/`mean_pl_loss` (E_pl/Ẽ_pl) |
| `evtools.display` | `repr_ansi`, `repr_plain`, `repr_html`, `repr_latex` |
| `evtools.conversions` | `mtob`, `mtopl`, `mtoq`, `btom`, ... + `conditioning_matrix`, `deconditioning_matrix`, `betp`, `plp` |
| `evtools.constants` | `ZERO_MASS`, `MASS_TOL`, `VALID_TOL`, `DISPLAY_TOL` |